# Visualize Excitation Rates
### Import libraries

In [ ]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.notebook import tqdm

In [ ]:
from utils import add_package_path

Paths

In [ ]:
# Current Working Directory
cwd = os.getcwd()
# WORKSPACE
ws = os.environ['WORKSPACE_PATH']

In [ ]:
prefix = '/home/zanardi/Workspace/ML/PyCOMET'
paths = {
    # PyCOMET source dir
    'pycomet': prefix + '/pycomet/',
    # SurQCT source dir
    'surqct': prefix + '/pycomet/examples/SurQCT/',
    # PyCOMET input module
    'inputs': prefix + '/run/SurQCT/O3_UMN/final/Diss/configs.ln/config.01/run.03/',
    # SurQCT input module
    'InputData': prefix + '/run/SurQCT/O3_UMN/final/Diss/datagen/genTrainData/'
}
add_package_path(paths)

In [ ]:
# Output directory
out = cwd + '/Output/DissRates/final.v1/'
if not os.path.exists(out):
    os.makedirs(out)

### Initialize PyCOMET model
Working case

In [ ]:
os.chdir(paths['inputs'])

Import additional modules

In [ ]:
from pycomet.utilities import utils
from pycomet.apps import functions as fun
# Inputs
import inputs
from inputs.general import flags

Compile model

In [ ]:
# Load network architecture
net = fun._net(
    input_mod=inputs
)
# Finalize model
for k in ('train', 'save'):
    flags[k] = 0
flags['load'] = 1
model = fun._model(
    flags=flags,
    input_mod=inputs,
    net=net
)

In [ ]:
os.chdir(cwd)

### Initialize SurQCT parameters

Import additional modules

In [ ]:
from surqct.src.chemical_systems import systems as syst
from surqct.src.rates_type.rates_type import RatesType
from surqct.src.utilities import utils
from surqct.src.utilities import const
from surqct import config as cfg
# SurQCT input module
import InputData

In [ ]:
matplotlib = utils.import_matplotlib()
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']
%matplotlib notebook

Inputs

In [ ]:
SuffixesHDF5 = {
    'QCT':    '/Orig/',
    'SurQCT': '/final.v1/'
}
TTranVec = [2500, 20000]
YLimTemp = [1.e-15, 1.e-12]
# TTranVec = [1500, 2500, 5000, 8000, 10000, 12000, 15000, 20000]
# YLimTemp = [1.e-15, 1.e-15, 1.e-14, 1.e-14, 1.e-14, 1.e-12, 1.e-12, 1.e-12]

Initializing QCT & SurQCT chemical systems

In [ ]:
Systems = {}
for Identifier, SuffixHDF5 in SuffixesHDF5.items():
    # Initializing rates
    Systems[Identifier] = syst.get_system(
        NameLong=InputData.SystemName,
        KinSource=Identifier,
        ProcNames=tf.nest.flatten(InputData.RatesType),
        PathToHDF5Fld=cfg.paths.HDF5Dtb + '/' + Identifier + '/' + SuffixHDF5
    )
# Number of levels
NLevels = list(Systems.values())[0].Processes.CompInOut[InputData.RatesType]['In'][0].NLevels

Reading rates & predicting rates for missing temperatures

In [ ]:
for Identifier, System in Systems.items():
    # Reading rates
    if os.path.exists(System.Processes.PathToHDF5File):
        System.Processes.read_rates(TTranVec=TTranVec)
    # Predicting rates for missing temperatures
    if Identifier == 'SurQCT':
        rates_type = RatesType(System, InputData)
        if System.Processes.Rates is None:
            System.Processes.Rates = {InputData.RatesType: {}}
        rates = System.Processes.Rates[InputData.RatesType]
        for TTran in TTranVec:
            TTranStr = str(int(TTran))
            if TTranStr not in rates.keys():
                xPred, yPred = rates_type.generate_predictiondata(
                    Model=model.net,
                    Columns=inputs.data.columns['x'],
                    TTranVec=tf.nest.flatten(TTran),
                    iIdxVec=np.arange(NLevels)
                )
                System.Processes.Rates[InputData.RatesType][TTranStr] = yPred.to_numpy()

### Plotting rates

In [ ]:
def plot_prediction2D(
    System,
    InputData,
    Path,
    PredData,
    TrueData=None,
    XAxis='EInt',
    FigName=None,
    ylim=[1.e-15, None]
):
    MolIn  = System.Processes.CompInOut[InputData.RatesType]['In'][0]
    
    # Rates and j-th level data
    if TrueData is not None:
        iIdxTrue, KTrue = TrueData
        XTrue = MolIn.DiatData[XAxis].to_numpy()[iIdxTrue]
        if XAxis == 'EInt':
            XTrue = XTrue - MolIn.DissE
    iIdxPred, KPred = PredData
    XPred = MolIn.DiatData[XAxis].to_numpy()[iIdxPred]
    if XAxis == 'EInt':
        XPred = XPred - MolIn.DissE
    
    XLbl = const.diat_data_lbl[XAxis]
    
    # Figure
    fig = plt.figure()
    if TrueData is not None:
        plt.scatter([], [], c=COLORS[0], label='QCT')
        plt.scatter(XTrue, KTrue, c=COLORS[0], s=10)
    plt.scatter([], [], c=COLORS[1], label='SurQCT')
    plt.scatter(XPred, KPred, c=COLORS[1], s=10, alpha=0.5)
    plt.yscale('log')
    plt.legend()
    plt.xlabel(XLbl)
    plt.ylabel(r'$K_{i}^D$ [cm$^3$/s]')
    plt.ylim(ylim)
    
    if FigName is None:
        FigName = 'Diss2D'
    plt.savefig(Path + '/' + FigName + '.pdf')

In [ ]:
for i, TTran in enumerate(tqdm(TTranVec)):
    TTranStr = str(int(TTran))
    NLevels  = list(Systems.values())[0].Processes.CompInOut[InputData.RatesType]['In'][0].NLevels
    X = np.arange(NLevels)
    # True data
    if TTranStr in Systems['QCT'].Processes.Rates[InputData.RatesType].keys():
        Y = Systems['QCT'].Processes.Rates[InputData.RatesType][TTranStr]
        TrueData = (X, Y)
    else:
        TrueData = None
    # Pred data
    Y = Systems['SurQCT'].Processes.Rates[InputData.RatesType][TTranStr]
    PredData = (X, Y)
    # Plotting
    plot_prediction2D(
        System=Systems['QCT'],
        InputData=InputData,
        Path=out,
        PredData=PredData,
        TrueData=TrueData,
        FigName=f'Diss2D_T{TTranStr}K',
        ylim=[YLimTemp[i], None]
    )

In [ ]:
import time
import os
import shutil
import sys
import h5py
import tensorflow                             as tf
import numpy                                  as np
import seaborn                                as sns
import pandas                                 as pd
import numpy.random                           as random
from sklearn.utils                        import shuffle
from sklearn.model_selection              import train_test_split
from tqdm                                 import tqdm
from os                                   import path
import pysftp                                 as sftp
import random                                 as rnd
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, zoomed_inset_axes


LocalFlg         = 'False'
if (not LocalFlg):
    WORKSPACE_PATH = "/home/venturi/WORKSPACE/"
    IP             = '128.174.245.172'
    User           = "venturi"
else:
    WORKSPACE_PATH = os.getenv('WORKSPACE_PATH')  
    
import matplotlib as mpl
import matplotlib.pyplot                      as plt

print("="*50)
print(" TensorFlow version: {}".format(tf.__version__))
print(" Eager execution: {}".format(tf.executing_eagerly()))

In [ ]:
pd.set_option('display.max_columns', None)  
sns.set_style('ticks')
plt.rc('text', usetex=True)
mpl.rcParams['text.latex.preamble'] = [r'\usepackage{amsmath} \boldmath']
# plt.rc('text.latex', preamble=r'\usepackage{pifont} \usepackage{tikz} \usepackage[utf8]{inputenc}')
plt.rc('font', family='Times')
sns.set_context("paper")

# Tick parameters
plt.rcParams['xtick.labelsize'] = 30
plt.rcParams['ytick.labelsize'] = 30
mpl.rcParams['xtick.major.size'] = 10
mpl.rcParams['xtick.major.width'] = 2
mpl.rcParams['xtick.minor.size'] = 10
mpl.rcParams['xtick.minor.width'] = 2
mpl.rcParams['ytick.major.size'] = 10
mpl.rcParams['ytick.major.width'] = 2
mpl.rcParams['ytick.minor.size'] = 10
mpl.rcParams['ytick.minor.width'] = 2

plt.rcParams['axes.labelsize'] = 30
plt.rcParams['axes.titlesize'] = 30
plt.rcParams['legend.fontsize'] = 30

In [ ]:
Dimension   = 'nondim'
InelRunNb   = 9
ExchRunNb   = 6
DissRunNb   = 55
ReactionFlgQCT    = '2_1_1_0'
ReactionFlgSurQCT = '2_1_1_0'
TTranVec = [5000, 8000, 10000, 12000, 20000]
PopPlotTime = [1e-6]

import Initialize_System
import Initialize_Paths as Paths

SystName    = 'O3_UMN'
Initialize = getattr(Initialize_System,'Initialize_'+SystName)
System = Initialize()

QCT_ME_path    = Paths.Workspace_path + 'Air_Database/' + System.QCTMEPath
SurQCT_ME_path = Paths.Workspace_path + 'Air_Database/Run_0D_surQCT/' + Dimension + '_RunI'+str(InelRunNb)+'_E'+str(ExchRunNb)+'_D'+str(DissRunNb)+'/'

Data_ME_path   = Paths.Surqct_path + '0D/Data/' + System.Name + '/' + Dimension + '_RunI'+str(InelRunNb)+'_E'+str(ExchRunNb)+'_D'+str(DissRunNb)+'/'
Figure_ME_path = Paths.Surqct_path + '0D/Figures/' + System.Name + '/' + Dimension + '_RunI'+str(InelRunNb)+'_E'+str(ExchRunNb)+'_D'+str(DissRunNb)+'/'

print('QCT Master Equation Path    :', QCT_ME_path)
print('SurQCT Master Equation Path :', SurQCT_ME_path)
print('Data Master Equation Path   :', Data_ME_path)
print('Figures Master Equation Path:', Figure_ME_path)

In [ ]:
SurQCTFldr          = Paths.Workspace_path + '/SurQCT/surqct/'

RatesType           = 'KExcit'
ExcitType           = 'KInel'
NNRunIdx            = 9

PathToRunFld        = SurQCTFldr + '/../' + RatesType + '/all_temperatures_nondim/' + ExcitType + '/' 

MinRate             = 1.e-15

iIdxVec             = np.array([110,3500], dtype=int)


# Reading Files
print("\n[SurQCT]: Loading Modules and Functions ...")

sys.path.insert(0, SurQCTFldr + '/src/Reading/')
from Reading  import read_levelsdata, read_diatdata

InputFld = PathToRunFld + '/Run_' + str(NNRunIdx) + '/'
print(InputFld)
sys.path.insert(0, InputFld)

# Initialize Input Data
print("\n[SurQCT]: Keep Loading Modules and Functions...")
from SurQCT_Input import inputdata

print("\n[SurQCT]: Initializing Input ...")
InputData    = inputdata(WORKSPACE_PATH, SurQCTFldr)

Prefix                    = 'Run_'
InputData.NNRunIdx        = NNRunIdx
InputData.PathToRunFld    = InputData.PathToRunFld+'/'+Prefix+str(InputData.NNRunIdx)
InputData.PathToFigFld    = InputData.PathToRunFld+'/'+InputData.PathToFigFld
InputData.PathToParamsFld = InputData.PathToRunFld+'/'+InputData.PathToParamsFld
InputData.PathToDataFld   = InputData.PathToRunFld+'/Data/'                                                               
InputData.PathToParamsFld = InputData.PathToRunFld+'/Params/'

# Read NN Model
print("\n[SurQCT]: Loading Final Modules ... ")

sys.path.insert(0, SurQCTFldr  + '/src/Model/' + InputData.ApproxModel + '/')
from Model import model


InputData.DefineModelIntFlg  = 0
InputData.TrainIntFlg        = 0
InputData.TransferFlg        = False
NN_KExcit                    = model(InputData, InputData.PathToRunFld, None, None)
NN_KExcit.load_params(InputData.PathToParamsFld)

OtherVar           = InputData.OtherVar
xVarsVec_i         = InputData.xVarsVec_i + ['vqn','jqn']
xVarsVec_Delta     = InputData.xVarsVec_Delta
xVarsVec           = list(set(xVarsVec_i) | set(xVarsVec_Delta))
print('[SurQCT]:   Reading Variables: ', xVarsVec)

InputData.MultFact = 1.e+9
MinValueTrain      = 1.e-16 * InputData.MultFact
MinValueTest       = 1.e-16 * InputData.MultFact
NoiseSD            = 1.e-13 * InputData.MultFact

NMolecules         = len(InputData.PathToLevelsFile)

InputData.iLevelsVecTest = list(np.array(InputData.iLevelsVecTest) - 1)

# Reading Levels Data
LevelsData = []
DiatData   = []
DiatData2  = []
NLevels    = []
for iMol in range(NMolecules):

    LevelsDataTemp = read_levelsdata(InputData.PathToLevelsFile[iMol], xVarsVec, '')
    LevelsData.append(LevelsDataTemp)

    DiatDataTemp = read_diatdata(InputData.PathToDiatFile[iMol], InputData.Molecules[iMol], TTranVec, TTranVec)
    DiatData.append(DiatDataTemp)
    
    NLevelsTemp    = LevelsDataTemp.shape[0]
    NLevels.append(NLevelsTemp)

In [ ]:
def read_HDF5(System, PathToHDF5File, TTra, TInt):                                                                                           
    print('[HCN]:   Reading Rates Data from: ' + PathToHDF5File)                                                                             
    HDF5Exist_Flg = path.exists(PathToHDF5File)                                                                                              
    if (HDF5Exist_Flg):                                                                                                                      
        f = h5py.File(PathToHDF5File, 'a')                                                                                                   
    else:                                                                                                                                    
        f = {'key': 'value'}                                                                                                                 
                                                                                                                                             
    try:                                                                                                                                     
        TStr  = 'T_' + str(int(TTra)) + '_' + str(int(TInt)) + '/Rates/'                                                                     
        grp   = f[TStr]                                                                                                                      
        Data  = grp["Diss"]                                                                                                                  
        System.Rates.KDiss = Data[...]                                                                                                       
        Data  = grp["Inel"]                                                                                                                  
        System.Rates.KInel = Data[...]                                                                                                       
                                                                                                                                             
        KExch = []                                                                                                                           
        for iProc in range(2, System.NbProcesses):                                                                                           
            ExchStr    = "Exch_" + str(iProc-1)                                                                                              
            Data       = grp[ExchStr]                                                                                                        
            System.Rates.KExch[iProc] = Data[...]                                                                                            
        f.close()                                                                                                                            
                                                                                                                                             
    except:                                                                                                                                  
        KDiss = None                                                                                                                         
                                                                                                                                             
    return System

In [ ]:
for TTran in TTranVec:
    Str = 'q_'+str(int(TTran))

    # Read Rates
    PathToHDF5File = System.PathToHDF5Fld+System.Name+'.hdf5'
    System = read_HDF5(System, PathToHDF5File, TTran, TTran)

    fig, axs = plt.subplots(2,2,sharey=True,sharex=True,figsize=(12,10))
    iplot = 0
    iplotc = 0
    for iIdx in tqdm(iIdxVec, desc='[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes'):
        iIdx -= 1
        print('Level: '+str(iIdx)+'; vqn: '+str(DiatData[0]['vqn'].to_numpy()[iIdx])+'; jqn:'+str(DiatData[0]['jqn'].to_numpy()[iIdx]))
        time.sleep(0.02)

        if (InputData.ExoEndoFlg):
            jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] <= DiatData[0]['EInt'].to_numpy()[iIdx])]
            jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
            RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
        else:
            jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    >= DiatData[0][Str].to_numpy()[iIdx])]
            jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
            RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
        jNLevels              = len(jIdxVec)
        jNLevelsNo            = len(jIdxVecNo)  

        ### FWD Rates
        iiIdxVec              = [iIdx]*jNLevels

        iTTranVec             = np.ones((jNLevels))*TTran
        TTranDataTemp         = pd.DataFrame({'TTran': iTTranVec})
        TTranDataTemp.index   = jIdxVec


        iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
        iLevelsDataTemp.index = jIdxVec

        jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        if (OtherVar == '_Delta'):
            jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
        else:
            jLevelsDataTemp   = jLevelsDataTemp
        jLevelsDataTemp.index = jIdxVec

        kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        kLevelsDataTemp.index = jIdxVec


        iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]

        jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
        jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

        kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]

        xTemp_FWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
            KExcit_NN_FWD          = np.exp( NN_KExcit.Model.predict(xTemp_FWD[NN_KExcit.xTrainingVar]) ) / InputData.MultFact 

        ## BWD Rates
        iiIdxVec              = jIdxVecNo
        jIdxVec               = [iIdx]*jNLevelsNo

        iTTranVec              = np.ones((jNLevelsNo))*TTran
        TTranDataTemp         = pd.DataFrame({'TTran': iTTranVec})
        TTranDataTemp.index   = iiIdxVec


        iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
        iLevelsDataTemp.index = iiIdxVec

        jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy() 
        jLevelsDataTemp.index = iiIdxVec
        if (OtherVar == '_Delta'):
            jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
        else:
            jLevelsDataTemp   = jLevelsDataTemp

        kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        kLevelsDataTemp.index = iiIdxVec


        iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]

        jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
        jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

        kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]

        xTempp_BWD            = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        if (len(xTempp_BWD[NN_KExcit.xTrainingVar]) > 0):
            KExcit_NN_BWD          = np.exp( NN_KExcit.Model.predict(xTempp_BWD[NN_KExcit.xTrainingVar]) ) / InputData.MultFact 
            KExcit_NN_BWD[:,0]     = KExcit_NN_BWD[:,0] / RatioNo

            iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
            iLevelsData.columns   = [(VarName + '_j') for VarName in iLevelsData.columns]

            jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
            jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

            kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
            kLevelsData.columns   = [(VarName + '_i') for VarName in kLevelsData.columns]

            xTemp_BWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        ### Concatenating
            if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
                xTemp     = pd.concat([xTemp_FWD, xTemp_BWD], axis=0)
                KExcit_NN = np.concatenate((KExcit_NN_FWD, KExcit_NN_BWD), axis=0)
            else:
                xTemp     = xTemp_BWD
                KExcit_NN = KExcit_NN_BWD
        else:
            if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
                xTemp     = xTemp_FWD
                KExcit_NN = KExcit_NN_FWD

        ### Plotting
        Mask     = System.Rates.KInel[iIdx,:] > MinRate

        vmax     = -8
        vmin     = -14

        cdict = {
          'red'  :  ( (0.0, 0.25, .25), (0.02, .59, .59), (1., 1., 1.)),
          'green':  ( (0.0, 0.0, 0.0), (0.02, .45, .45), (1., .97, .97)),
          'blue' :  ( (0.0, 1.0, 1.0), (0.02, .75, .75), (1., 0.45, 0.45))
        }
        
        axs[iplotc][iplot].scatter(DiatData[1]['vqn'][Mask], DiatData[1]['jqn'][Mask],s=100, c=np.log10(System.Rates.KInel[iIdx,Mask]), vmin=vmin, vmax=vmax,cmap='jet')
        axs[iplotc][iplot].set_xlim([DiatData[1]['vqn'].min(), DiatData[1]['vqn'].max()])
        axs[iplotc][iplot].set_ylim([DiatData[1]['jqn'].min(), 200])

        sc = axs[iplotc][iplot+1].scatter(xTemp['vqn_j'], xTemp['jqn_j'], c=np.log10(KExcit_NN[:,0]),s=100, vmin=vmin, vmax=vmax,cmap='jet')
        axs[iplotc][iplot+1].set_xlim([DiatData[1]['vqn'].min(), DiatData[1]['vqn'].max()])
        axs[iplotc][iplot+1].set_ylim([DiatData[1]['jqn'].min(), 200])
        axs[iplotc][iplot+1].set_xticks([0,10,20,30,40])
        axs[iplotc][iplot+1].set_yticks([0,25,50,75,100,125,150,175])

        iplotc = iplotc + 1
    
    fig.add_subplot(111, frame_on=False)
    plt.tick_params(labelcolor="none", bottom=False, left=False)
    plt.xticks([])
    plt.yticks([])
    
    plt.xlabel('\\textbf{vqn}',labelpad=40)
    plt.ylabel('\\textbf{jqn}',labelpad=60)
    
#     fig.subplots_adjust(right=0.8)
    fig.subplots_adjust(wspace=0.1, hspace=0.1, right=0.8)
    cbar_ax = fig.add_axes([0.82, 0.15, 0.03, 0.7])
    cbar = fig.colorbar(sc, cax=cbar_ax)
    cbar.set_label('$\log_{10}{K_{Inel}}$ \\textbf{[}$cm^3/s$\\textbf{]}', rotation=90,labelpad=25,size=30)

#     fig.tight_layout(pad=2.0) 
#     plt.savefig((Figure_ME_path+System.Name+'_T'+str(TTran)+'K_KInel.pdf'), format='pdf',dpi=600)


# Plot on Diatomic Potential

In [ ]:
iIdxVec             = np.array([110,3500, 5000, 1200, 6000, 2392], dtype=int)

for TTran in TTranVec:
    Str = 'q_'+str(int(TTran))

    # Read Rates
    PathToHDF5File = System.PathToHDF5Fld+System.Name+'.hdf5'
    System = read_HDF5(System, PathToHDF5File, TTran, TTran)

    fig, axs = plt.subplots(1,1,figsize=(10,5))
    sc = axs.scatter(DiatData[1]['jqn'], DiatData[1]['EInt'],s=50, c='grey',alpha=0.2)
    
    for iIdx in tqdm(iIdxVec, desc='[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes'):
        iIdx -= 1
        print('Level: '+str(iIdx)+'; vqn: '+str(DiatData[0]['vqn'].to_numpy()[iIdx])+'; jqn:'+str(DiatData[0]['jqn'].to_numpy()[iIdx]))
        time.sleep(0.02)

 
        ### Plotting
        Mask     = System.Rates.KInel[iIdx,:] > 5e-13

        vmax     = -9
        vmin     = -13
        
        sc = axs.scatter(DiatData[1]['jqn'][Mask], DiatData[1]['EInt'][Mask],s=100, c=np.log10(System.Rates.KInel[iIdx,Mask]), vmin=vmin, vmax=vmax,cmap='jet')
    
    axs.set_xlim([DiatData[1]['jqn'].min(), DiatData[1]['jqn'].max()])
#     axs.set_ylim([DiatData[1]['jqn'].min(), 200])
#     axs.set_xticks([0,10,20,30,40])
#     axs.set_yticks([0,25,50,75,100,125,150,175])

    axins1 = inset_axes(axs,
                        width="50%",  # width = 50% of parent_bbox width
                        height="5%",  # height : 5%
                        loc='upper left')
    
    cbar = fig.colorbar(sc, cax=axins1, orientation="horizontal", ticks=[-12,-11,-10,-9])
    axins1.xaxis.set_ticks_position("bottom")    
    cbar.set_label('$\log_{10}{K_{Inel}}$ \\textbf{[}$cm^3/s$\\textbf{]}',fontsize=20)

    fig.add_subplot(111, frame_on=False)
    plt.tick_params(labelcolor="none", bottom=False, left=False)
    plt.xticks([])
    plt.yticks([])
    
    plt.xlabel('\\textbf{jqn}',labelpad=40)
    plt.ylabel('$\epsilon$ \\textbf{[eV]}',labelpad=80)
    
    
    fig.tight_layout(pad=2.0) 
    plt.savefig(('/home/venturi/WORKSPACE/SurQCT/0D/Figures/OUSD_poster/T'+str(TTran)+'K_KInel.pdf'), format='pdf',dpi=600)


## Correlation plot 

In [ ]:
SaveFigures = False

In [ ]:
NIdx = 100
KExcit_Scatter = []
sampled_levels = []
KExcitNN_Scatter = []

for TTran in TTranVec:
    Str = 'q_'+str(int(TTran))

    print('[SurQCT]:    TTran = ', TTran)
    iIdxVec = np.arange(NLevels[0])
    sampled_levels_temp = rnd.choices(list(iIdxVec), k=NIdx)
    
    # Read Rates
    PathToHDF5File = System.PathToHDF5Fld+System.Name+'.hdf5'
    PathToHDF5File = '/home/venturi/WORKSPACE/Air_Database/HDF5_Database_AmalInel/'+System.Name+'.hdf5'
    System = read_HDF5(System, PathToHDF5File, TTran, TTran)
    
    for iIdx in sampled_levels_temp:
        iIdx -= 1
#         print('Level: '+str(iIdx)+'; vqn: '+str(DiatData[0]['vqn'].to_numpy()[iIdx])+'; jqn:'+str(DiatData[0]['jqn'].to_numpy()[iIdx]))
        time.sleep(0.02)

        if (InputData.ExoEndoFlg):
            jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] <= DiatData[0]['EInt'].to_numpy()[iIdx])]
            jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
            RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
        else:
            jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    >= DiatData[0][Str].to_numpy()[iIdx])]
            jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
            RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
        jNLevels              = len(jIdxVec)
        jNLevelsNo            = len(jIdxVecNo)  

        ### FWD Rates
        iiIdxVec              = [iIdx]*jNLevels

        iTTranVec              = np.ones((jNLevels))*TTran
        TTranDataTemp         = pd.DataFrame({'TTran': iTTranVec})
        TTranDataTemp.index   = jIdxVec


        iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
        iLevelsDataTemp.index = jIdxVec

        jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        if (OtherVar == '_Delta'):
            jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
        else:
            jLevelsDataTemp   = jLevelsDataTemp
        jLevelsDataTemp.index = jIdxVec

        kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        kLevelsDataTemp.index = jIdxVec


        iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]

        jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
        jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

        kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]

        xTemp_FWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
            KExcit_NN_FWD          = np.exp( NN_KExcit.Model.predict(xTemp_FWD[NN_KExcit.xTrainingVar]) ) / InputData.MultFact
       
        ## BWD Rates
        iiIdxVec              = jIdxVecNo
        jIdxVec               = [iIdx]*jNLevelsNo

        iTTranVec              = np.ones((jNLevelsNo))*TTran
        TTranDataTemp         = pd.DataFrame({'TTran': iTTranVec})
        TTranDataTemp.index   = iiIdxVec


        iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
        iLevelsDataTemp.index = iiIdxVec

        jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy() 
        jLevelsDataTemp.index = iiIdxVec
        if (OtherVar == '_Delta'):
            jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
        else:
            jLevelsDataTemp   = jLevelsDataTemp

        kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        kLevelsDataTemp.index = iiIdxVec


        iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]

        jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
        jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

        kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]

        xTempp_BWD            = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        if (len(xTempp_BWD[NN_KExcit.xTrainingVar]) > 0):
            KExcit_NN_BWD          = np.exp( NN_KExcit.Model.predict(xTempp_BWD[NN_KExcit.xTrainingVar]) ) / InputData.MultFact 
            KExcit_NN_BWD[:,0]     = KExcit_NN_BWD[:,0] / RatioNo

            iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
            iLevelsData.columns   = [(VarName + '_j') for VarName in iLevelsData.columns]

            jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
            jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

            kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
            kLevelsData.columns   = [(VarName + '_i') for VarName in kLevelsData.columns]

            xTemp_BWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        ### Concatenating
            if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
                xTemp     = pd.concat([xTemp_FWD, xTemp_BWD], axis=0)
                KExcit_NN = np.concatenate((KExcit_NN_FWD, KExcit_NN_BWD), axis=0)
            else:
                xTemp     = xTemp_BWD
                KExcit_NN = KExcit_NN_BWD
        else:
            if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
                xTemp     = xTemp_FWD
                KExcit_NN = KExcit_NN_FWD

        sampled_levels_temp1 = rnd.choices(list(iIdxVec), k=100)
    
        KExcit_Scatter.append(System.Rates.KInel[iIdx,sampled_levels_temp1])
        KExcitNN_Scatter.append(KExcit_NN[sampled_levels_temp1,0])

#         System.Rates.KInel[iIdx,iIdx] = 0.0
#         KExcit_NN[iIdx,0] = 0.0
#         KExcit_Scatter.append(np.sum(System.Rates.KInel[iIdx,:]))
#         KExcitNN_Scatter.append(np.sum(KExcit_NN[sampled_levels_temp1,0]))
    
#     ax.plot([1e-13,1e-8],[1e-13,1e-8],'r',linewidth=3.0)
#     # axis properties
#     plt.xscale('log')
#     plt.yscale('log')
#     plt.xlim([1.e-13, 5.e-9]) 
#     plt.ylim([1.e-13, 5.e-9])  
#     plt.xticks([1e-13,1e-11,1e-9])
#     plt.yticks([1e-13,1e-11,1e-9])

#     plt.xlabel('$k^D_{i,true}$ \\textbf{[}$cm^3/s$\\textbf{]}')
#     plt.ylabel('$k^D_{i,pred}$ \\textbf{[}$cm^3/s$\\textbf{]}')
#     plt.minorticks_off()
    
#     aspectratio=1.0
#     ax.set_aspect(1.0/1.0, adjustable='box')
#     fig.tight_layout(pad=0.5) 
#     if(SaveFigures):
#         plt.savefig((Figure_ME_path+System.Name+'_T'+str(TTran)+'K_KDissScatter.pdf'), format='pdf',dpi=300)
        


In [ ]:
SaveFigures = False
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator,LogLocator)

fig,ax = plt.subplots(1,1,figsize=(8,8),sharey=True)
ax.scatter(KExcit_Scatter, KExcitNN_Scatter, s=20, c='k',label='\\textbf{QCT}')

ax.plot([2e-15,2e-8],[1e-15,1e-8],'r',linewidth=3.0)
# axis properties
plt.xscale('log')
plt.yscale('log')
plt.xlim([1.e-12, 1.e-10]) 
plt.ylim([1.e-12, 1.e-10])  
plt.xticks([1e-12,1e-11])
plt.yticks([1e-12,1e-11])

plt.xlabel('$k^D_{i,true}$ \\textbf{[}$cm^3/s$\\textbf{]}')
plt.ylabel('$k^D_{i,pred}$ \\textbf{[}$cm^3/s$\\textbf{]}')
plt.minorticks_off()

aspectratio=1.0
ax.set_aspect(1.0/1.0, adjustable='box')
fig.tight_layout(pad=0.5) 
if(SaveFigures):
    plt.savefig((Figure_ME_path+System.Name+'_T'+str(TTran)+'K_KDissScatter.pdf'), format='pdf',dpi=300)

# N3 Plots

In [ ]:
SystName    = 'N3_NASA'
Initialize = getattr(Initialize_System,'Initialize_'+SystName)
System = Initialize()

QCT_ME_path    = Paths.Workspace_path + 'Air_Database/' + System.QCTMEPath
SurQCT_ME_path = Paths.Workspace_path + 'Air_Database/Run_0D_surQCT/' + Dimension + '_RunI'+str(InelRunNb)+'_E'+str(ExchRunNb)+'_D'+str(DissRunNb)+'/'

Data_ME_path   = Paths.Surqct_path + '0D/Data/' + System.Name + '/' + Dimension + '_RunI'+str(InelRunNb)+'_E'+str(ExchRunNb)+'_D'+str(DissRunNb)+'/'
Figure_ME_path = Paths.Surqct_path + '0D/Figures/' + System.Name + '/' + Dimension + '_RunI'+str(InelRunNb)+'_E'+str(ExchRunNb)+'_D'+str(DissRunNb)+'/'

print('QCT Master Equation Path    :', QCT_ME_path)
print('SurQCT Master Equation Path :', SurQCT_ME_path)
print('Data Master Equation Path   :', Data_ME_path)
print('Figures Master Equation Path:', Figure_ME_path)

In [ ]:
WORKSPACE_PATH      = os.environ['WORKSPACE_PATH']
SurQCTFldr          = WORKSPACE_PATH + '/SurQCT/surqct/'

RatesType           = 'KExcit'
ExcitType           = 'KInel'
NNRunIdx            = 9

PathToRunFld        = SurQCTFldr + '/../' + RatesType + '/all_temperatures_nondim/' + ExcitType + '/' 

TTranVec            = [10000,20000]
MinRate             = 1.e-15

# iIdxVec             = np.array([1, 10, 34, 35, 81, 82, 98, 1000, 2000, 3000, 4581], dtype=int)
iIdxVec             = np.array([110,3500], dtype=int)


# Reading Files
print("\n[SurQCT]: Loading Modules and Functions ...")

sys.path.insert(0, SurQCTFldr  + '/src/Reading/')
from Reading  import read_levelsdata, read_diatdata

InputFld = PathToRunFld + '/Run_' + str(NNRunIdx) + '/'
print(InputFld)
sys.path.insert(0, InputFld)

# Initialize Input Data
print("\n[SurQCT]: Keep Loading Modules and Functions...")
from SurQCT_Input import inputdata

print("\n[SurQCT]: Initializing Input ...")
InputData    = inputdata(WORKSPACE_PATH, SurQCTFldr)

Prefix                    = 'Run_'
InputData.NNRunIdx        = NNRunIdx
InputData.PathToRunFld    = InputData.PathToRunFld+'/'+Prefix+str(InputData.NNRunIdx)
InputData.PathToFigFld    = InputData.PathToRunFld+'/'+InputData.PathToFigFld
InputData.PathToParamsFld = InputData.PathToRunFld+'/'+InputData.PathToParamsFld
InputData.PathToDataFld   = InputData.PathToRunFld+'/Data/'                                                               
InputData.PathToParamsFld = InputData.PathToRunFld+'/Params/'

InputData.PathToHDF5File  = InputData.WORKSPACE_PATH  + '/Air_Database/HDF5_Database_semiClassicalApprox/N3_NASA.hdf5'
InputData.Molecules       = ['N2','N2'] 
InputData.PathToLevelsFile= [InputData.WORKSPACE_PATH + '/Air_Database/Run_0D/database/levels/N2_LeRoy_log_nd.csv',
                             InputData.WORKSPACE_PATH + '/Air_Database/Run_0D/database/levels/N2_LeRoy_log_nd.csv']
InputData.PathToDiatFile  = [InputData.WORKSPACE_PATH + '/CoarseAIR/coarseair/dtb/Molecules/N2/LeRoy/MyLeroy_FromRobyn.inp',
                            InputData.WORKSPACE_PATH + '/CoarseAIR/coarseair/dtb/Molecules/N2/LeRoy/MyLeroy_FromRobyn.inp']   

# Read NN Model
print("\n[SurQCT]: Loading Final Modules ... ")

sys.path.insert(0, SurQCTFldr  + '/src/Model/' + InputData.ApproxModel + '/')
from Model import model


InputData.DefineModelIntFlg  = 0
InputData.TrainIntFlg        = 0
InputData.TransferFlg        = False
NN_KExcit                    = model(InputData, InputData.PathToRunFld, None, None)
NN_KExcit.load_params(InputData.PathToParamsFld)

OtherVar           = InputData.OtherVar
xVarsVec_i         = InputData.xVarsVec_i + ['vqn','jqn']
xVarsVec_Delta     = InputData.xVarsVec_Delta
xVarsVec           = list(set(xVarsVec_i) | set(xVarsVec_Delta))
print('[SurQCT]:   Reading Variables: ', xVarsVec)

InputData.MultFact = 1.e+9
MinValueTrain      = 1.e-16 * InputData.MultFact
MinValueTest       = 1.e-16 * InputData.MultFact
NoiseSD            = 1.e-13 * InputData.MultFact

NMolecules         = len(InputData.PathToLevelsFile)

InputData.iLevelsVecTest = list(np.array(InputData.iLevelsVecTest) - 1)

# Reading Levels Data
LevelsData = []
DiatData   = []
DiatData2  = []
NLevels    = []
for iMol in range(NMolecules):

    LevelsDataTemp = read_levelsdata(InputData.PathToLevelsFile[iMol], xVarsVec, '')
    LevelsData.append(LevelsDataTemp)

    DiatDataTemp = read_diatdata(InputData.PathToDiatFile[iMol], InputData.Molecules[iMol], TTranVec, TTranVec)
    DiatData.append(DiatDataTemp)
    
    NLevelsTemp    = LevelsDataTemp.shape[0]
    NLevels.append(NLevelsTemp)
    


In [ ]:
# Read Rates
PathToHDF5File = System.PathToHDF5Fld+System.Name+'.hdf5'
System = read_HDF5(System, PathToHDF5File, TTran, TTran)

In [ ]:
for TTran in TTranVec:
    Str = 'q_'+str(int(TTran))

    # Read Rates
    PathToHDF5File = System.PathToHDF5Fld+System.Name+'.hdf5'
    System = read_HDF5(System, PathToHDF5File, TTran, TTran)

    fig, axs = plt.subplots(2,2,sharey=True,sharex=True,figsize=(12,10))
    iplot = 0
    iplotc = 0
    for iIdx in tqdm(iIdxVec, desc='[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes'):
        iIdx -= 1
        print('Level: '+str(iIdx)+'; vqn: '+str(DiatData[0]['vqn'].to_numpy()[iIdx])+'; jqn:'+str(DiatData[0]['jqn'].to_numpy()[iIdx]))
        time.sleep(0.02)

        if (InputData.ExoEndoFlg):
            jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] <= DiatData[0]['EInt'].to_numpy()[iIdx])]
            jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
            RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
        else:
            jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    >= DiatData[0][Str].to_numpy()[iIdx])]
            jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
            RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
        jNLevels              = len(jIdxVec)
        jNLevelsNo            = len(jIdxVecNo)  

        ### FWD Rates
        iiIdxVec              = [iIdx]*jNLevels

        iTTranVec              = np.ones((jNLevels))*TTran
        TTranDataTemp         = pd.DataFrame({'TTran': iTTranVec})
        TTranDataTemp.index   = jIdxVec


        iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
        iLevelsDataTemp.index = jIdxVec

        jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        if (OtherVar == '_Delta'):
            jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
        else:
            jLevelsDataTemp   = jLevelsDataTemp
        jLevelsDataTemp.index = jIdxVec

        kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        kLevelsDataTemp.index = jIdxVec


        iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]

        jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
        jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

        kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]

        xTemp_FWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
            KExcit_NN_FWD          = np.exp( NN_KExcit.Model.predict(xTemp_FWD[NN_KExcit.xTrainingVar]) ) / InputData.MultFact 

        ## BWD Rates
        iiIdxVec              = jIdxVecNo
        jIdxVec               = [iIdx]*jNLevelsNo

        iTTranVec              = np.ones((jNLevelsNo))*TTran
        TTranDataTemp         = pd.DataFrame({'TTran': iTTranVec})
        TTranDataTemp.index   = iiIdxVec


        iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
        iLevelsDataTemp.index = iiIdxVec

        jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy() 
        jLevelsDataTemp.index = iiIdxVec
        if (OtherVar == '_Delta'):
            jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
        else:
            jLevelsDataTemp   = jLevelsDataTemp

        kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        kLevelsDataTemp.index = iiIdxVec


        iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]

        jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
        jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

        kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]

        xTempp_BWD            = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        if (len(xTempp_BWD[NN_KExcit.xTrainingVar]) > 0):
            KExcit_NN_BWD          = np.exp( NN_KExcit.Model.predict(xTempp_BWD[NN_KExcit.xTrainingVar]) ) / InputData.MultFact 
            KExcit_NN_BWD[:,0]     = KExcit_NN_BWD[:,0] / RatioNo

            iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
            iLevelsData.columns   = [(VarName + '_j') for VarName in iLevelsData.columns]

            jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
            jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

            kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
            kLevelsData.columns   = [(VarName + '_i') for VarName in kLevelsData.columns]

            xTemp_BWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        ### Concatenating
            if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
                xTemp     = pd.concat([xTemp_FWD, xTemp_BWD], axis=0)
                KExcit_NN = np.concatenate((KExcit_NN_FWD, KExcit_NN_BWD), axis=0)
            else:
                xTemp     = xTemp_BWD
                KExcit_NN = KExcit_NN_BWD
        else:
            if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
                xTemp     = xTemp_FWD
                KExcit_NN = KExcit_NN_FWD

        ### Plotting
        Mask     = System.Rates.KInel[iIdx,:] > MinRate

        vmax     = -8
        vmin     = -14

        cdict = {
          'red'  :  ( (0.0, 0.25, .25), (0.02, .59, .59), (1., 1., 1.)),
          'green':  ( (0.0, 0.0, 0.0), (0.02, .45, .45), (1., .97, .97)),
          'blue' :  ( (0.0, 1.0, 1.0), (0.02, .75, .75), (1., 0.45, 0.45))
        }
        
        axs[iplotc][iplot].scatter(DiatData[1]['vqn'][Mask], DiatData[1]['jqn'][Mask],s=100, c=np.log10(System.Rates.KInel[iIdx,Mask]), vmin=vmin, vmax=vmax,cmap='jet')
        axs[iplotc][iplot].set_xlim([DiatData[1]['vqn'].min(), DiatData[1]['vqn'].max()])
        axs[iplotc][iplot].set_ylim([DiatData[1]['jqn'].min(), 200])

        sc = axs[iplotc][iplot+1].scatter(xTemp['vqn_j'], xTemp['jqn_j'], c=np.log10(KExcit_NN[:,0]),s=100, vmin=vmin, vmax=vmax,cmap='jet')
        axs[iplotc][iplot+1].set_xlim([DiatData[1]['vqn'].min(), DiatData[1]['vqn'].max()])
        axs[iplotc][iplot+1].set_ylim([DiatData[1]['jqn'].min(), 200])
        axs[iplotc][iplot+1].set_xticks([0,10,20,30,40])
        axs[iplotc][iplot+1].set_yticks([0,25,50,75,100,125,150,175])

        iplotc = iplotc + 1
    
    fig.add_subplot(111, frame_on=False)
    plt.tick_params(labelcolor="none", bottom=False, left=False)
    plt.xticks([])
    plt.yticks([])
    
    plt.xlabel('\\textbf{vqn}',labelpad=40)
    plt.ylabel('\\textbf{jqn}',labelpad=60)
    
#     fig.subplots_adjust(right=0.8)
    fig.subplots_adjust(wspace=0.1, hspace=0.1, right=0.8)
    cbar_ax = fig.add_axes([0.82, 0.15, 0.03, 0.7])
    cbar = fig.colorbar(sc, cax=cbar_ax)
    cbar.set_label('$\log_{10}{K_{Inel}}$ \\textbf{[}$cm^3/s$\\textbf{]}', rotation=90,labelpad=25,size=30)

#     fig.tight_layout(pad=2.0) 
#     plt.savefig((Figure_ME_path+System.Name+'_T'+str(TTran)+'K_KInel.pdf'), format='pdf',dpi=600)


In [ ]:
# iIdxVec             = np.array([110,3500, 7400, 1200, 9000, 2392], dtype=int)

iIdxVec             = np.array([1026, 6100, 7200, 1200, 9200, 4518], dtype=int)

for TTran in TTranVec:
    Str = 'q_'+str(int(TTran))

    # Read Rates
    PathToHDF5File = System.PathToHDF5Fld+System.Name+'.hdf5'
    System = read_HDF5(System, PathToHDF5File, TTran, TTran)

    fig, axs = plt.subplots(1,1,figsize=(10,5))
    sc = axs.scatter(DiatData[1]['jqn'], DiatData[1]['EInt'],s=50, c='grey',alpha=0.2)
    
    for iIdx in tqdm(iIdxVec, desc='[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes'):
        iIdx -= 1
        print('Level: '+str(iIdx)+'; vqn: '+str(DiatData[0]['vqn'].to_numpy()[iIdx])+'; jqn:'+str(DiatData[0]['jqn'].to_numpy()[iIdx]))
        time.sleep(0.02)

 
        ### Plotting
        Mask     = System.Rates.KInel[iIdx,:] > 5e-13

        vmax     = -9
        vmin     = -13
        
        sc = axs.scatter(DiatData[1]['jqn'][Mask], DiatData[1]['EInt'][Mask],s=100, c=np.log10(System.Rates.KInel[iIdx,Mask]), vmin=vmin, vmax=vmax,cmap='jet')
    
    axs.set_xlim([DiatData[1]['jqn'].min(), DiatData[1]['jqn'].max()])
#     axs.set_ylim([DiatData[1]['jqn'].min(), 200])
#     axs.set_xticks([0,10,20,30,40])
#     axs.set_yticks([0,25,50,75,100,125,150,175])

    axins1 = inset_axes(axs,
                        width="50%",  # width = 50% of parent_bbox width
                        height="5%",  # height : 5%
                        loc='upper left')
    
    cbar = fig.colorbar(sc, cax=axins1, orientation="horizontal", ticks=[-12,-11,-10,-9])
    axins1.xaxis.set_ticks_position("bottom")    
    cbar.set_label('$\log_{10}{K_{Inel}}$ \\textbf{[}$cm^3/s$\\textbf{]}',fontsize=20)

    fig.add_subplot(111, frame_on=False)
    plt.tick_params(labelcolor="none", bottom=False, left=False)
    plt.xticks([])
    plt.yticks([])
    
    plt.xlabel('\\textbf{jqn}',labelpad=40)
    plt.ylabel('$\epsilon$ \\textbf{[eV]}',labelpad=80)
    
    
    fig.tight_layout(pad=2.0) 
    plt.savefig(('/home/venturi/WORKSPACE/SurQCT/0D/Figures/OUSD_poster/N2_T'+str(TTran)+'K_KInel.pdf'), format='pdf',dpi=600)


# NON Plots

In [ ]:
SystName    = 'NON_UMN'
Initialize = getattr(Initialize_System,'Initialize_'+SystName)
System = Initialize()

QCT_ME_path    = Paths.Workspace_path + 'Air_Database/' + System.QCTMEPath
SurQCT_ME_path = Paths.Workspace_path + 'Air_Database/Run_0D_surQCT/' + Dimension + '_RunI'+str(InelRunNb)+'_E'+str(ExchRunNb)+'_D'+str(DissRunNb)+'/'

Data_ME_path   = Paths.Surqct_path + '0D/Data/' + System.Name + '/' + Dimension + '_RunI'+str(InelRunNb)+'_E'+str(ExchRunNb)+'_D'+str(DissRunNb)+'/'
Figure_ME_path = Paths.Surqct_path + '0D/Figures/' + System.Name + '/' + Dimension + '_RunI'+str(InelRunNb)+'_E'+str(ExchRunNb)+'_D'+str(DissRunNb)+'/'

print('QCT Master Equation Path    :', QCT_ME_path)
print('SurQCT Master Equation Path :', SurQCT_ME_path)
print('Data Master Equation Path   :', Data_ME_path)
print('Figures Master Equation Path:', Figure_ME_path)

In [ ]:
SurQCTFldr          = Paths.Workspace_path + '/SurQCT/surqct/'
RatesType           = 'KExcit'
ExcitType           = 'KInel'
NNRunIdx            = 9

PathToRunFld        = SurQCTFldr + '/../' + RatesType + '/all_temperatures_nondim/' + ExcitType + '/' 

TTranVec            = [10000]
MinRate             = 1.e-15

iIdxVec             = np.array([50,2000], dtype=int)


# Reading Files
print("\n[SurQCT]: Loading Modules and Functions ...")

sys.path.insert(0, SurQCTFldr  + '/src/Reading/')
from Reading  import read_levelsdata, read_diatdata

InputFld = PathToRunFld + '/Run_' + str(NNRunIdx) + '/'
print(InputFld)
sys.path.insert(0, InputFld)

# Initialize Input Data
print("\n[SurQCT]: Keep Loading Modules and Functions...")
from SurQCT_Input import inputdata

print("\n[SurQCT]: Initializing Input ...")
InputData    = inputdata(WORKSPACE_PATH, SurQCTFldr)

Prefix                    = 'Run_'
InputData.NNRunIdx        = NNRunIdx
InputData.PathToRunFld    = InputData.PathToRunFld+'/'+Prefix+str(InputData.NNRunIdx)
InputData.PathToFigFld    = InputData.PathToRunFld+'/'+InputData.PathToFigFld
InputData.PathToParamsFld = InputData.PathToRunFld+'/'+InputData.PathToParamsFld
InputData.PathToDataFld   = InputData.PathToRunFld+'/Data/'                                                               
InputData.PathToParamsFld = InputData.PathToRunFld+'/Params/'

InputData.PathToHDF5File  = InputData.WORKSPACE_PATH  + '/Air_Database/HDF5_Database/NON_UMN.hdf5'
InputData.Molecules       = ['NO','NO'] 
InputData.PathToLevelsFile= [InputData.WORKSPACE_PATH + '/Air_Database/Run_0D/database/levels/NO_UMN_log_nd.csv',
                             InputData.WORKSPACE_PATH + '/Air_Database/Run_0D/database/levels/NO_UMN_log_nd.csv']
InputData.PathToDiatFile  = [InputData.WORKSPACE_PATH + '/CoarseAIR/coarseair/dtb/Molecules/NO/UMN/Recomputed.inp',
                             InputData.WORKSPACE_PATH + '/CoarseAIR/coarseair/dtb/Molecules/NO/UMN/Recomputed.inp'] 

# Read NN Model
print("\n[SurQCT]: Loading Final Modules ... ")

sys.path.insert(0, SurQCTFldr  + '/src/Model/' + InputData.ApproxModel + '/')
from Model import model


InputData.DefineModelIntFlg  = 0
InputData.TrainIntFlg        = 0
InputData.TransferFlg        = False
NN_KExcit                    = model(InputData, InputData.PathToRunFld, None, None)
NN_KExcit.load_params(InputData.PathToParamsFld)

OtherVar           = InputData.OtherVar
xVarsVec_i         = InputData.xVarsVec_i + ['vqn','jqn']
xVarsVec_Delta     = InputData.xVarsVec_Delta
xVarsVec           = list(set(xVarsVec_i) | set(xVarsVec_Delta))
print('[SurQCT]:   Reading Variables: ', xVarsVec)

InputData.MultFact = 1.e+9
MinValueTrain      = 1.e-16 * InputData.MultFact
MinValueTest       = 1.e-16 * InputData.MultFact
NoiseSD            = 1.e-13 * InputData.MultFact

NMolecules         = len(InputData.PathToLevelsFile)

InputData.iLevelsVecTest = list(np.array(InputData.iLevelsVecTest) - 1)

# Reading Levels Data
LevelsData = []
DiatData   = []
DiatData2  = []
NLevels    = []
for iMol in range(NMolecules):

    LevelsDataTemp = read_levelsdata(InputData.PathToLevelsFile[iMol], xVarsVec, '')
    LevelsData.append(LevelsDataTemp)

    DiatDataTemp = read_diatdata(InputData.PathToDiatFile[iMol], InputData.Molecules[iMol], TTranVec, TTranVec)
    DiatData.append(DiatDataTemp)
    
    NLevelsTemp    = LevelsDataTemp.shape[0]
    NLevels.append(NLevelsTemp)
    


In [ ]:
# Read Rates
PathToHDF5File = System.PathToHDF5Fld+System.Name+'.hdf5'
System = read_HDF5(System, PathToHDF5File, TTran, TTran)

In [ ]:
for TTran in TTranVec:
    Str = 'q_'+str(int(TTran))

    # Read Rates
    PathToHDF5File = System.PathToHDF5Fld+System.Name+'.hdf5'
    System = read_HDF5(System, PathToHDF5File, TTran, TTran)

    fig, axs = plt.subplots(2,2,sharey=True,sharex=True,figsize=(12,10))
    iplot = 0
    iplotc = 0
    for iIdx in tqdm(iIdxVec, desc='[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes'):
        iIdx -= 1
        print('Level: '+str(iIdx)+'; vqn: '+str(DiatData[0]['vqn'].to_numpy()[iIdx])+'; jqn:'+str(DiatData[0]['jqn'].to_numpy()[iIdx]))
        time.sleep(0.02)

        if (InputData.ExoEndoFlg):
            jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] <= DiatData[0]['EInt'].to_numpy()[iIdx])]
            jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
            RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1]['EInt'].to_numpy()[jIdx] > DiatData[0]['EInt'].to_numpy()[iIdx])]
        else:
            jIdxVec           = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    >= DiatData[0][Str].to_numpy()[iIdx])]
            jIdxVecNo         = [jIdx for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
            RatioNo           = [DiatData[0][Str].to_numpy()[iIdx]/DiatData[1][Str].to_numpy()[jIdx] for jIdx in np.arange(NLevels[1]) if (DiatData[1][Str].to_numpy()[jIdx]    < DiatData[0][Str].to_numpy()[iIdx])]
        jNLevels              = len(jIdxVec)
        jNLevelsNo            = len(jIdxVecNo)  

        ### FWD Rates
        iiIdxVec              = [iIdx]*jNLevels

        iTTranVec              = np.ones((jNLevels))*TTran
        TTranDataTemp         = pd.DataFrame({'TTran': iTTranVec})
        TTranDataTemp.index   = jIdxVec


        iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
        iLevelsDataTemp.index = jIdxVec

        jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        if (OtherVar == '_Delta'):
            jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
        else:
            jLevelsDataTemp   = jLevelsDataTemp
        jLevelsDataTemp.index = jIdxVec

        kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        kLevelsDataTemp.index = jIdxVec


        iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]

        jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
        jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

        kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]

        xTemp_FWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
            KExcit_NN_FWD          = np.exp( NN_KExcit.Model.predict(xTemp_FWD[NN_KExcit.xTrainingVar]) ) / InputData.MultFact 

        ## BWD Rates
        iiIdxVec              = jIdxVecNo
        jIdxVec               = [iIdx]*jNLevelsNo

        iTTranVec              = np.ones((jNLevelsNo))*TTran
        TTranDataTemp         = pd.DataFrame({'TTran': iTTranVec})
        TTranDataTemp.index   = iiIdxVec


        iLevelsDataTemp       = LevelsData[0].iloc[iiIdxVec,:].copy()
        iLevelsDataTemp.index = iiIdxVec

        jLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy() 
        jLevelsDataTemp.index = iiIdxVec
        if (OtherVar == '_Delta'):
            jLevelsDataTemp   = iLevelsDataTemp.subtract(jLevelsDataTemp) 
        else:
            jLevelsDataTemp   = jLevelsDataTemp

        kLevelsDataTemp       = LevelsData[1].iloc[jIdxVec,:].copy()        
        kLevelsDataTemp.index = iiIdxVec


        iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        iLevelsData.columns   = [(VarName + '_i') for VarName in iLevelsData.columns]

        jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
        jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

        kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
        kLevelsData.columns   = [(VarName + '_j') for VarName in kLevelsData.columns]

        xTempp_BWD            = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        if (len(xTempp_BWD[NN_KExcit.xTrainingVar]) > 0):
            KExcit_NN_BWD          = np.exp( NN_KExcit.Model.predict(xTempp_BWD[NN_KExcit.xTrainingVar]) ) / InputData.MultFact 
            KExcit_NN_BWD[:,0]     = KExcit_NN_BWD[:,0] / RatioNo

            iLevelsData           = pd.concat([iLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
            iLevelsData.columns   = [(VarName + '_j') for VarName in iLevelsData.columns]

            jLevelsData           = pd.concat([jLevelsDataTemp[xVarsVec_Delta], TTranDataTemp], axis=1)
            jLevelsData.columns   = [(VarName + OtherVar) for VarName in jLevelsData.columns]

            kLevelsData           = pd.concat([kLevelsDataTemp[xVarsVec_i], TTranDataTemp], axis=1)
            kLevelsData.columns   = [(VarName + '_i') for VarName in kLevelsData.columns]

            xTemp_BWD             = pd.concat([iLevelsData, jLevelsData, kLevelsData], axis=1)

        ### Concatenating
            if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
                xTemp     = pd.concat([xTemp_FWD, xTemp_BWD], axis=0)
                KExcit_NN = np.concatenate((KExcit_NN_FWD, KExcit_NN_BWD), axis=0)
            else:
                xTemp     = xTemp_BWD
                KExcit_NN = KExcit_NN_BWD
        else:
            if (len(xTemp_FWD[NN_KExcit.xTrainingVar]) > 0):
                xTemp     = xTemp_FWD
                KExcit_NN = KExcit_NN_FWD

        ### Plotting
        Mask     = System.Rates.KInel[iIdx,:] > MinRate

        vmax     = -8
        vmin     = -14

        cdict = {
          'red'  :  ( (0.0, 0.25, .25), (0.02, .59, .59), (1., 1., 1.)),
          'green':  ( (0.0, 0.0, 0.0), (0.02, .45, .45), (1., .97, .97)),
          'blue' :  ( (0.0, 1.0, 1.0), (0.02, .75, .75), (1., 0.45, 0.45))
        }
        
        axs[iplotc][iplot].scatter(DiatData[1]['vqn'][Mask], DiatData[1]['jqn'][Mask],s=100, c=np.log10(System.Rates.KInel[iIdx,Mask]), vmin=vmin, vmax=vmax,cmap='jet')
        axs[iplotc][iplot].set_xlim([DiatData[1]['vqn'].min(), DiatData[1]['vqn'].max()])
        axs[iplotc][iplot].set_ylim([DiatData[1]['jqn'].min(), 200])

        sc = axs[iplotc][iplot+1].scatter(xTemp['vqn_j'], xTemp['jqn_j'], c=np.log10(KExcit_NN[:,0]),s=100, vmin=vmin, vmax=vmax,cmap='jet')
        axs[iplotc][iplot+1].set_xlim([DiatData[1]['vqn'].min(), DiatData[1]['vqn'].max()])
        axs[iplotc][iplot+1].set_ylim([DiatData[1]['jqn'].min(), 200])
        axs[iplotc][iplot+1].set_xticks([0,10,20,30,40])
        axs[iplotc][iplot+1].set_yticks([0,25,50,75,100,125,150,175])

        iplotc = iplotc + 1
    
    fig.add_subplot(111, frame_on=False)
    plt.tick_params(labelcolor="none", bottom=False, left=False)
    plt.xticks([])
    plt.yticks([])
    
    plt.xlabel('\\textbf{vqn}',labelpad=40)
    plt.ylabel('\\textbf{jqn}',labelpad=60)
    
#     fig.subplots_adjust(right=0.8)
    fig.subplots_adjust(wspace=0.1, hspace=0.1, right=0.8)
    cbar_ax = fig.add_axes([0.82, 0.15, 0.03, 0.7])
    cbar = fig.colorbar(sc, cax=cbar_ax)
    cbar.set_label('$\log_{10}{K_{Inel}}$ \\textbf{[}$cm^3/s$\\textbf{]}', rotation=90,labelpad=25,size=30)

#     fig.tight_layout(pad=2.0) 
    plt.savefig((Figure_ME_path+System.Name+'_T'+str(TTran)+'K_KInel.pdf'), format='pdf',dpi=600)


In [ ]:
# iIdxVec             = np.array([110,3500, 7400, 1200, 9000, 2392], dtype=int)

iIdxVec             = np.array([797, 2827, 4660, 1200, 6600, 4518], dtype=int)

for TTran in TTranVec:
    Str = 'q_'+str(int(TTran))

    # Read Rates
    PathToHDF5File = System.PathToHDF5Fld+System.Name+'.hdf5'
    System = read_HDF5(System, PathToHDF5File, TTran, TTran)

    fig, axs = plt.subplots(1,1,figsize=(10,5))
    sc = axs.scatter(DiatData[1]['jqn'], DiatData[1]['EInt'],s=50, c='grey',alpha=0.2)
    
    for iIdx in tqdm(iIdxVec, desc='[SurQCT]:     Generating Inelastic and Exchange Rate Matrixes'):
        iIdx -= 1
        print('Level: '+str(iIdx)+'; vqn: '+str(DiatData[0]['vqn'].to_numpy()[iIdx])+'; jqn:'+str(DiatData[0]['jqn'].to_numpy()[iIdx]))
        time.sleep(0.02)

 
        ### Plotting
        Mask     = System.Rates.KInel[iIdx,:] > 5e-13

        vmax     = -9
        vmin     = -13
        
        sc = axs.scatter(DiatData[1]['jqn'][Mask], DiatData[1]['EInt'][Mask],s=100, c=np.log10(System.Rates.KInel[iIdx,Mask]), vmin=vmin, vmax=vmax,cmap='jet')
    
    axs.set_xlim([DiatData[1]['jqn'].min(), DiatData[1]['jqn'].max()])
#     axs.set_ylim([DiatData[1]['jqn'].min(), 200])
#     axs.set_xticks([0,10,20,30,40])
#     axs.set_yticks([0,25,50,75,100,125,150,175])

    axins1 = inset_axes(axs,
                        width="50%",  # width = 50% of parent_bbox width
                        height="5%",  # height : 5%
                        loc='upper left')
    
    cbar = fig.colorbar(sc, cax=axins1, orientation="horizontal", ticks=[-12,-11,-10,-9])
    axins1.xaxis.set_ticks_position("bottom")    
    cbar.set_label('$\log_{10}{K_{Inel}}$ \\textbf{[}$cm^3/s$\\textbf{]}',fontsize=20)

    fig.add_subplot(111, frame_on=False)
    plt.tick_params(labelcolor="none", bottom=False, left=False)
    plt.xticks([])
    plt.yticks([])
    
    plt.xlabel('\\textbf{jqn}',labelpad=40)
    plt.ylabel('$\epsilon$ \\textbf{[eV]}',labelpad=80)
    
    
    fig.tight_layout(pad=2.0) 
    plt.savefig(('/home/venturi/WORKSPACE/SurQCT/0D/Figures/OUSD_poster/NO_T'+str(TTran)+'K_KInel.pdf'), format='pdf',dpi=600)


In [ ]:
print(NLevels)